In [33]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready')

Libraries ready


In [34]:
from groq import Groq
API_KEY="XXXX"
client=Groq(api_key=API_KEY)
MODEL='llama-3.1-8b-instant'
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [35]:
def ask_llm(
    user_message,
    system_message="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=1500 # user(trainer) defined words namma kudukarathu thaan
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content
test_response = ask_llm(
    "What is Catharanthus roseus? Answer in exactly 2 sentences."
)

print("=== LLM Response ===")
print(test_response)

=== LLM Response ===
Catharanthus roseus, also known as Madagascar periwinkle, is a flowering plant native to the island of Madagascar, and it is widely cultivated for its ornamental value and its unique, trumpet-shaped flowers. This plant is also known for its medicinal properties, particularly its alkaloids vincristine and vinblastine, which are used in chemotherapy to treat various types of cancer.


In [36]:
response_etl=ask_llm(# user message
    "In 3 bullet points,explain the chemical components in Catharanthus roseus"
    "How Catharanthus roseus used in medical field",
    system_message="You are a Biology Student"#system message ,system's role to get relevant answer
    "Be consise and stay desire of learning new"
)
print("=== LLM Response ===")
print("Catharanthus roseus")
print(response_etl)
print()
print('Token Explaination')
print('Each word is roughly 1-2 tokens')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')


=== LLM Response ===
Catharanthus roseus
As a biology student, I'd be happy to provide you with information on Catharanthus roseus. Here are three bullet points on its chemical components and its uses in the medical field:

**Chemical Components:**

* **Alkaloids**: Catharanthus roseus contains several alkaloids, including vinblastine, vincristine, and ajmalicine, which have been isolated and used in medicine.
* **Flavonoids**: The plant is also a source of flavonoids, such as kaempferol and quercetin, which have been shown to have antioxidant and anti-inflammatory properties.
* **Glycosides**: Catharanthus roseus contains glycosides, including ajugoside and aucubin, which have been reported to have antidiabetic and antimicrobial activities.

**Uses in the Medical Field:**

* **Cancer Treatment**: The alkaloids vinblastine and vincristine, isolated from Catharanthus roseus, are used in chemotherapy to treat various types of cancer, including leukemia and lymphoma.
* **Anti-HIV Medicati

In [37]:
response_etl=ask_llm(
    "What is Data Science?"
    "You are a Biology Student",
    system_message="You are a Biology Student"
    "Be consise and stay desire of learning new"
)
print("=== LLM Response ===")
print("Catharanthus roseus")
print(response_etl)
print()
print('Token Explaination')
print('Each word is roughly 1-2 tokens')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')


=== LLM Response ===
Catharanthus roseus
As a Biology student, I've had exposure to data analysis in lab settings, but Data Science is a broader field that combines statistics, computer science, and domain expertise (in this case, biology) to extract insights from complex data.

Data Science involves:

1. **Data collection**: Gathering data from various sources (e.g., experiments, surveys, sensors).
2. **Data cleaning and preprocessing**: Ensuring data quality and transforming it into a suitable format for analysis.
3. **Data analysis**: Using statistical and machine learning techniques to identify patterns, trends, and correlations within the data.
4. **Data visualization**: Presenting findings in a clear and concise manner using graphs, charts, and other visualizations.
5. **Modeling and prediction**: Developing predictive models to forecast future outcomes or simulate hypothetical scenarios.

In biology, Data Science applications include:

1. **Genomics**: Analyzing large DNA and RN

In [38]:
#Cleaning the data
zero_shot_response=ask_llm(
    "Extract the city name from this address:"
    "456 Brigade Road,Bangalore 560025,Karnataka,India"
)
print("Zero Shot LLM response:")
print(zero_shot_response)
print()
ambiguous_response=ask_llm("Clean this data:ramesh kumar,45000,mumbai")
print(ambiguous_response)
print()
print('Problem:Output format is unpredictable and not machine-parseable!!')

Zero Shot LLM response:
The city name is: Bangalore.

The data appears to be in a simple format. To clean it, I'll assume that each line of output represents a person with their name, salary, and location. Based on this assumption, here's a cleaned-up version:

1. Name: Ramesh Kumar
2. Salary: 45000
3. Location: Mumbai

If there are multiple lines of data, I can help you process them as well. However, please provide the data in a single line or multiple lines for me to clean.

Here's the cleaned-up version of the data in a structured format:

Name  | Salary | Location
------|--------|---------
Ramesh Kumar | 45000 | Mumbai

Problem:Output format is unpredictable and not machine-parseable!!


In [39]:
few_shot_prompt="""
Convert employee text to JSON.Here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"Ramesh Kumar","salary":45000,"city":"MUMBAI"}
Input:HARSHINI,80000,COIMBATORE
Output:{"name":"Harshini","salary":80000,"city":"Coimbatore"}
Input:Bala Kumar,80000,COIMBATORE
Output:
"""
few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print("Few Shot LLM response:")
print(few_shot_response)
print()
try:
  parsed=json.loads(few_shot_response.strip())
  print("Successfully parsed JSON!!")
  print(f"Name:{parsed['name']}")
  print(f"Salary:{parsed['salary']}")
  print(f"City:{parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed --> model added extra text")
  print("Solution:add explicit instructions in the system prompt")

Few Shot LLM response:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def employee_to_json(employee_text):
    """
    Convert employee text to JSON.

    Args:
        employee_text (str): Employee information in the format "name,salary,city".

    Returns:
        dict: Employee information in JSON format.
    """
    # Split the employee text into individual fields
    fields = employee_text.split(',')

    # Capitalize the first letter of each field and convert to title case
    name = fields[0].title()
    salary = fields[1]
    city = fields[2].upper()

    # Create a dictionary with the employee information
    employee_info = {
        "name": name,
        "salary": int(salary),
        "city": city
    }

    # Convert the dictionary to JSON
    employee_json = json.dumps(employee_info)

    return employee_json

# Test the function
print(employee_to_json("RAMESH KUMAR,45000,mumbai"))
print(employee_to_json("HARSHINI,80000,COIMBAT

In [40]:
## few shot prompt
#boy girl determination
few_shot_prompt="""
Tells the gender of a person by name,

Input :Harshini
Output:Female

Input:Bala Kumar
Output:Male

Input:Kanishka
Output:Female

Input:Sandhiya
Output:
State:Tamil Nadu

who is the Sandhiya?
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.8)
print("==== Few Shot Result====")
print(few_shot_response,'\n')

try:
  parsed = json.loads(few_shot_response.strip())
  print("Successfully parsed as JSON")
  print(f"Name: {parsed['name']}, Salary: {parsed['salary']}, City: {parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed - model added extra text")
  print("Solution: add explicit instructions in the system prompt")

==== Few Shot Result====
Based on the information provided, I will try to identify the gender of the person by name.

1. Harshini - Female
2. Bala Kumar - Male
3. Kanishka - Female (Note: Although Kanishka is typically a male name in Indian culture, it can also be used as a female name in some regions, especially in southern India. However, without further information, it's difficult to pinpoint the exact gender.)
4. Sandhiya - I couldn't find any information on a well-known person named Sandhiya. However, it seems that you provided additional information about her being from Tamil Nadu.

After conducting a quick search, I found that Sandhiya is an Indian actress who primarily works in Tamil cinema. Based on this information, I would assume that the Sandhiya you are referring to is an actress from Tamil Nadu.

So, to answer your question: 
Output: Female (Assuming the Sandhiya you are referring to is the Indian actress from Tamil Nadu.) 

Parsing failed - model added extra text
Solutio

In [41]:
same_question="Review this Python code and identify any issues:\n" \
              "df['revenue']=df['qty'] =df['price]\n" \
              "result=df.groupby(qty,price)"
generic_response=ask_llm(same_question,temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()
role_response=ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production "
    "experience.Review code critically for production readiness, "
    "data type issues,and potential failures at scale.",
temperature=0.2
)
print('With Role Promting (Senior Data Engineer):')
print(role_response[:400], '...')
print()
print('Notice: role prompting produces more technical, actionable feedback')

Without Role Prompting:
There are several issues with the provided Python code:

1.  **Assignment Operator**: The line `df['revenue']=df['qty'] =df['price']` is using a single equals sign `=` which is the assignment operator. It's trying to assign the value of `df['price']` to both `df['revenue']` and `df['qty']`. However, ...

With Role Promting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a snippet from a larger data analysis or data engineering project. However, there are several issues that need to be addressed to make it production-ready.

```python
df['revenue']=df['qty'] =df['price']
result=df.groupby(qty,price)
```

**Issues:**

1. **Assignment Operator**: The code uses a single equals sign (`=`) for assignment, which is ...

Notice: role prompting produces more technical, actionable feedback


In [42]:
#temperature experiment
prompt="Give me one creative name for a data analytics startup."
print(' ===Temperature Experiment=== ')
for temp in [0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature{temp}:{response.strip()}')
  time.sleep(1)

print()
print("Observations:")
print('Temperature=0.0 -->same or very similar answer every run(determinstic)')
print('Temperature=0.5 -->some variation')
print('Temperature=1.0 -->more creative/varied,sometimes surprising')
print()
print('Rule for data engineering tasks:use temperature=0.0 or 0.1')
print('You need CONSISTENT,PARSABLE output -->not creative variation')

 ===Temperature Experiment=== 
Temperature0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature0.5:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" implies connection and intersection of data points, while "Insights" conveys the idea of gaining valuable knowledge and understanding from the data. The name suggests a company that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature1.0:One creative name for a data analytics startup could be "NexaPulse". This name suggests the connection and rhythm of data analysis, implying a startup that excels at finding insights and trends in complex datasets.

Observations:
Temperat

In [43]:
messy_invoices=[
    "INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45000 Laptop purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for office Cleaning services",
    "#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024|Python training",
    "SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20",
    "Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95000|Server hardware"
]
print('Messy Invoices to process:')
for i,inv in enumerate(messy_invoices,1):
  print(f'{i}.{inv}')
print(f'\n Total :{len(messy_invoices)} invoices')

Messy Invoices to process:
1.INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45000 Laptop purchase
2.Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for office Cleaning services
3.#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024|Python training
4.SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20
5.Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95000|Server hardware

 Total :5 invoices


In [44]:
EXTRACTION_SYSTEM_PROMPT = """
You are an expert invoice extraction system.
Extract invoice number, vendor name, date, amount, and description.
Return only valid JSON.
"""
print("Extraction prompt engineered successfully!")
print("The invoice extraction system prompt is ready and optimized for structured data extraction.\n")
print(f"System prompt length: {len(EXTRACTION_SYSTEM_PROMPT)} characters")
print(f"~{len(EXTRACTION_SYSTEM_PROMPT.split())} words, ~{int(len(EXTRACTION_SYSTEM_PROMPT.split()) * 1.3)} tokens")

Extraction prompt engineered successfully!
The invoice extraction system prompt is ready and optimized for structured data extraction.

System prompt length: 138 characters
~20 words, ~26 tokens


In [45]:
def extract_invoice_data(invoice_text,system_prompt,client,model):
    """
    Extract structured JSON from a single messy invoice string.
    Returns a Python dict or None on failure.
    """
    try:
        response=client.chat.completions.create(
            model=model,
            messages=[
                {"role":"system","content":system_prompt},
                {"role":"user","content":f"Extract from: {invoice_text}"}
            ],
            temperature=0.0,
            max_tokens=400
        )
        raw = response.choices[0].message.content.strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            pass
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        print(f"WARNING: Could not parse JSON from response: {raw[:80]}...")
        return None
    except Exception as e:
        print(f"Error calling API: {e}")
        return None
print("Processing invoices with LLM...")
extracted_records = []
for i, invoice in enumerate(messy_invoices, 1):
    print(f"\n[{i}/{len(messy_invoices)}]Input:{invoice[:70]}...")
    result=extract_invoice_data(
        invoice,EXTRACTION_SYSTEM_PROMPT,client,
        MODEL
    )
    if result:
        extracted_records.append(result)
        print(
            f"Extracted Vendor={result.get('vendor_name')}, "
            f"Amount={result.get('amount')}, "
            f"Date={result.get('invoice_date')}"
        )
    else:
        print("--> Failed, adding placeholder")
        extracted_records.append({
            "invoice_id":None,"vendor_name":"EXTRACTION_FAILED","amount":None,"currency":"INR","invoice_date":None,"category":"Other","description":invoice[:50]
        })
    time.sleep(0.5)
print(f"\nProcessed:{len(extracted_records)}/{len(messy_invoices)} invoices")

Processing invoices with LLM...

[1/5]Input:INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45000 Laptop purcha...
Extracted Vendor=TECHWORLD SOLUTIONS, Amount=45000, Date=None

[2/5]Input:Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt:12500 for office Clea...
Extracted Vendor=PRIYA ENTERPRISES, Amount=12500, Date=None

[3/5]Input:#INV-2024-103 | arjun nair consultancy | 8000 | march 15 2024|Python t...
Extracted Vendor=arjun nair consultancy, Amount=8000, Date=None

[4/5]Input:SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01...
Extracted Vendor=SURESH RAO HARDWARE STORE, Amount=25000, Date=None

[5/5]Input:Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95...
Extracted Vendor=Ananya Tech Solutions, Amount=95000, Date=None

Processed:5/5 invoices


In [46]:
invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(
    invoices_df['amount'],
    errors='coerce'
)
invoices_df['invoice_date'] = pd.to_datetime(
    invoices_df['date'],
    errors='coerce'
)
invoices_df = invoices_df.drop(columns=['date'])
print("SMART DATA CLEANER OUTPUT")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}")
print()
print(invoices_df.to_string(index=False))

SMART DATA CLEANER OUTPUT
Rows: 5 | Columns: 5

                invoice_number               vendor_name  amount                    description invoice_date
                 INV-2024-0891       TECHWORLD SOLUTIONS   45000                Laptop purchase   2024-01-15
Invoice from PRIYA ENTERPRISES         PRIYA ENTERPRISES   12500       office Cleaning services          NaT
                 #INV-2024-103    arjun nair consultancy    8000                Python training          NaT
                          None SURESH RAO HARDWARE STORE   25000 Keyboard and Mouse accessories          NaT
                       INV-897     Ananya Tech Solutions   95000                Server hardware          NaT


In [47]:
print("Processing Invoices with LLM...")
user_invoice_message = "\n".join(messy_invoices)
llm_raw_response = ask_llm(user_invoice_message,
                            system_message='''Clean the data and organize data in same structure
                            (company,amount, invoice_date(YYYY-MM-DD), product)
                            return as JSON.
                            Only output the JSON.
                            No code.''')
print("==== Raw LLM Response ====")
print(llm_raw_response,'\n')
try:
  extracted_records = json.loads(llm_raw_response)
  print("Successfully parsed LLM response to a list of dictionaries.")
except json.JSONDecodeError as e:
  print(f"Error decoding JSON from LLM response: {e}")
  print("LLM Raw Response (unparseable):\n", llm_raw_response)
  extracted_records = []
except Exception as e:
  print(f"An unexpected error occurred during parsing: {e}")
  extracted_records = []

Processing Invoices with LLM...
==== Raw LLM Response ====
[
  {
    "company": "Techworld Solutions",
    "amount": 45000,
    "invoice_date": "2024-01-15",
    "product": "Laptop purchase"
  },
  {
    "company": "Priya Enterprises",
    "amount": 12500,
    "invoice_date": "2024-02-07",
    "product": "Office Cleaning services"
  },
  {
    "company": "Arjun Nair Consultancy",
    "amount": 8000,
    "invoice_date": "2024-03-15",
    "product": "Python training"
  },
  {
    "company": "Suresh Rao Hardware Store",
    "amount": 25000,
    "invoice_date": "2024-01-20",
    "product": "Keyboard and Mouse accessories"
  },
  {
    "company": "Ananya Tech Solutions",
    "amount": 95000,
    "invoice_date": "2024-02-28",
    "product": "Server hardware"
  }
] 

Successfully parsed LLM response to a list of dictionaries.


In [48]:
invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(invoices_df['amount'],errors='coerce')
invoices_df['invoice_date'] = pd.to_datetime(invoices_df['invoice_date'],errors='coerce')
print("=== SMART DATA CLEANER OUTPUT ===")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}\n")
print(invoices_df.to_string(index=False))

=== SMART DATA CLEANER OUTPUT ===
Rows: 5 | Columns: 4

                  company  amount invoice_date                        product
      Techworld Solutions   45000   2024-01-15                Laptop purchase
        Priya Enterprises   12500   2024-02-07       Office Cleaning services
   Arjun Nair Consultancy    8000   2024-03-15                Python training
Suresh Rao Hardware Store   25000   2024-01-20 Keyboard and Mouse accessories
    Ananya Tech Solutions   95000   2024-02-28                Server hardware
